# 🧭 팀 프로젝트 · 데이터분석 노트북 템플릿
> 우리 팀이 정한 주제로 **문제 정의부터 모델 활용까지** 한 바퀴 도는 노트북. 7~9차시 프로젝트와 같은 순서로 진행한다.

| 단계 | 할 일 | 보고서 |
|---|---|---|
| ① 문제 정의 | 주제·배경·분석 질문·X와 y 정하기 | 1장 |
| ② 데이터 수집 | 출처 기록 · 불러오기 · 구조 확인 | 2장 |
| ③ 탐색 및 전처리 | 결측·중복·이상치 처리 · 분포와 관계 탐색 · 파생 변수 | 3장 |
| ④ 모델 생성 및 학습 | 기준선 · 첫 모델 학습 | 4장 |
| ⑤ 모델 평가 | 회귀: R²·MAE / 분류: 정확도·혼동 행렬·정밀도·재현율 | 5장 |
| ⑥ 성능 개선 | 특성 바꾸기 · 모델 바꾸기 · 교차 검증으로 비교 | 6장 |
| ⑦ 모델 활용 | 새 데이터 예측 · 저장 · 활용 제안 · 한계와 윤리 | 7장 |

**사용법**
1. **⚙️ 설정 셀**에서 파일 이름·정답 열·문제 종류·특성만 우리 팀 데이터에 맞게 바꾼다.
2. 위에서부터 차례로 실행한다. `✍️ 우리 팀의 기록` 칸에 관찰과 해석을 적는다.
3. `🔧` 표시 셀은 우리 데이터에 맞게 고쳐 쓴다.
4. 마지막 **⑧ 보고서용 요약**의 표를 보고서 템플릿에 옮긴다.

> 처음에는 예시 데이터(`bike_2025.csv`: 공공자전거 하루 대여건수, 회귀 · 수업 연습용 가상 데이터)로 설정되어 있어 그대로 실행된다. 먼저 한 번 실행해 흐름을 본 뒤 우리 데이터로 바꾸자.
>
> 분류 예시로 연습하려면 설정을 `DATA_FILE='resume_screening.csv'`, `TARGET='Recruiter Decision'`, `TASK='분류'`, `FEATURES=['Experience (Years)', 'Projects Count']`, `DROP_COLS=['Resume_ID', 'Name', 'AI Score (0-100)']`로 바꾼다 (9차시 채용 데이터).

## 0. 준비 — 도구 불러오기

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (r2_score, mean_absolute_error, accuracy_score,
                             confusion_matrix, classification_report)
plt.rcParams['figure.figsize'] = (6, 3.5)
DATA = 'data' if os.path.exists('data') else '../data'

## ⚙️ 설정 — 우리 팀 데이터에 맞게 바꾸는 곳
| 변수 | 뜻 | 예시 |
|---|---|---|
| `DATA_FILE` | `data/` 폴더 안의 파일 이름 | `'bike_2025.csv'` |
| `ENCODING` | 글자 형식. 한글이 깨지면 `'cp949'` | `'utf-8-sig'` |
| `TARGET` | 맞히고 싶은 **정답 열 y** | `'대여건수'` |
| `TASK` | 정답이 숫자면 `'회귀'`, 종류면 `'분류'` | `'회귀'` |
| `FEATURES` | 첫 모델에 쓸 **특성 X** (2~3개로 시작) | `['평균기온', '강수량']` |
| `DROP_COLS` | 분석에 쓰지 않을 열 (ID, 이름 등) | `['날짜']` |

In [ ]:
# ⚙️ 🔧 우리 팀 설정 ----------------------------------------------
TEAM = '○○팀'                       # 팀 이름
DATA_FILE = 'bike_2025.csv'          # data/ 폴더 안 파일 이름
ENCODING = 'utf-8-sig'               # 한글이 깨지면 'cp949'
TARGET = '대여건수'                   # 정답 열(y)
TASK = '회귀'                         # '회귀' 또는 '분류'
FEATURES = ['평균기온', '강수량']      # 첫 모델에 쓸 특성(X)
DROP_COLS = ['날짜']                  # 쓰지 않을 열 (없으면 [])
RANDOM_STATE = 42                     # 결과를 똑같이 다시 얻기 위한 번호
# ---------------------------------------------------------------
assert TASK in ['회귀', '분류'], "TASK는 '회귀' 또는 '분류'로 적어 주세요."
print(f'{TEAM} · {DATA_FILE} · 정답: {TARGET} · 문제: {TASK} · 첫 특성: {FEATURES}')

### 🧰 도구 상자 — 수정하지 않아도 되는 함수
글자(범주) 열은 원-핫 인코딩, 숫자 열은 표준화를 자동으로 해 주고, 3-2에서 미처 처리하지 못한 결측치도 임시로 채워 주는 모델 조립 함수와 평가 함수다. (6~9차시에 배운 내용을 묶어 둔 것)

In [ ]:
def build_model(algorithm, features, data, degree=2):
    """algorithm: 'linear'(선형/로지스틱) · 'poly'(다항, 회귀만) · 'tree'(결정 트리) · 'linear_balanced'(분류, 불균형 보정)"""
    num = [c for c in features if pd.api.types.is_numeric_dtype(data[c])]
    cat = [c for c in features if c not in num]
    parts = []
    if num:   # 숫자: 빈칸은 중앙값으로 임시로 채우고(안전장치) 표준화
        parts.append((make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num))
    if cat:   # 글자: 빈칸은 가장 많은 값으로 채우고 원-핫 인코딩
        parts.append((make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat))
    prep = make_column_transformer(*parts)
    if TASK == '회귀':
        if algorithm == 'poly':
            return make_pipeline(prep, PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())
        if algorithm == 'tree':
            return make_pipeline(prep, DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE))
        return make_pipeline(prep, LinearRegression())
    if algorithm == 'tree':
        return make_pipeline(prep, DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE))
    weight = 'balanced' if algorithm == 'linear_balanced' else None
    return make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight=weight, random_state=RANDOM_STATE))


def score(model, X_te, y_te):
    """회귀: R²·MAE / 분류: 정확도·불균형 확인용 F1(두 클래스 평균)"""
    pred = model.predict(X_te)
    if TASK == '회귀':
        return {'R²': round(r2_score(y_te, pred), 3), 'MAE': round(mean_absolute_error(y_te, pred), 3)}
    from sklearn.metrics import f1_score
    return {'정확도': round(accuracy_score(y_te, pred), 3), 'F1(평균)': round(f1_score(y_te, pred, average='macro'), 3)}


def cv_score(model, X, y):
    """5겹 교차 검증 평균 점수 (회귀: R², 분류: F1 평균)"""
    if TASK == '회귀':
        cv, metric = KFold(5, shuffle=True, random_state=RANDOM_STATE), 'r2'
    else:
        cv, metric = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), 'f1_macro'
    return round(cross_val_score(model, X, y, cv=cv, scoring=metric).mean(), 3)

print('도구 상자 준비 완료')

---
# ① 문제 정의
### 주제 선정 기준 (네 가지 모두 '예'인 주제를 고른다)
- [ ] **궁금한가?** 우리 팀이 정말 알고 싶은 질문인가
- [ ] **데이터가 있는가?** 공개 데이터로 구할 수 있는가 (최소 수십~수백 행)
- [ ] **예측할 수 있는가?** 맞히고 싶은 정답(y)이 데이터에 열로 있는가
- [ ] **쓸모가 있는가?** 결과를 쓸 사람과 쓰임새가 떠오르는가

### 좋은 분석 질문 만들기
`궁금증` → `질문` → `분석 질문(기간·대상·측정 기준이 분명)`
예) 자전거가 궁금하다 → 날씨가 대여에 영향을 줄까? → **2025년 서울 공공자전거 하루 대여건수를 기온과 강수량으로 예측할 수 있을까?**

✍️ **우리 팀의 문제 정의**

| 항목 | 내용 |
|---|---|
| 주제 |  |
| 배경 (왜 궁금한가) |  |
| 분석 질문 |  |
| 정답 y (맞힐 것) |  |
| 특성 X 후보 |  |
| 문제 종류 (회귀/분류)와 이유 |  |
| 누가, 어디에 쓸까 |  |

---
# ② 데이터 수집
### 공개 데이터를 찾을 수 있는 곳
| 사이트 | 특징 |
|---|---|
| 공공데이터포털 (data.go.kr) | 정부·공공기관 데이터 |
| 국가통계포털 KOSIS (kosis.kr) | 국가 통계표 (7차시 흡연 데이터) |
| 기상자료개방포털 (data.kma.go.kr) | 날씨 관측 자료 (8차시 성산 기상 자료) |
| 서울 열린데이터광장 (data.seoul.go.kr) | 서울시 데이터 (공공자전거 등) |
| Kaggle (kaggle.com) | 전 세계 연습용 공개 데이터 (9차시 채용 데이터) |

> 내려받은 파일은 `data/` 폴더에 넣고, **⚙️ 설정**의 `DATA_FILE`을 바꾼다. 원본 파일은 고치지 말고 그대로 보관한다.

✍️ **데이터 출처 기록**

| 항목 | 내용 |
|---|---|
| 데이터 이름 |  |
| 출처 (사이트·기관) |  |
| 수집 기간 / 내려받은 날짜 |  |
| 한 행의 의미 (예: 하루, 한 사람) |  |
| 이용 조건 (공공누리 등) |  |

In [ ]:
# ▶️ 데이터 불러오기
df_raw = pd.read_csv(f'{DATA}/{DATA_FILE}', encoding=ENCODING)
print('(행, 열) =', df_raw.shape)
df_raw.head()

In [ ]:
# ▶️ 열 이름과 자료형 확인 — 숫자가 object(글자)로 읽힌 열은 없는지 살핀다 (7차시)
df_raw.info()

---
# ③ 탐색 및 전처리
## 3-1. 쓰지 않을 열 지우기 · 기초 통계

In [ ]:
df = df_raw.drop(columns=DROP_COLS, errors='ignore').copy()
df.describe(include='all').T

## 3-2. 결측치 — 개수를 세고, **이유에 맞게** 처리한다 (8차시)

In [ ]:
na = df.isna().sum()
print(na[na > 0] if na.sum() > 0 else '결측치 없음')

In [ ]:
# 🔧 결측치 처리 — 우리 데이터에 맞게 골라 쓰기 (필요한 줄의 # 을 지운다)
# df = df.dropna(subset=[TARGET])                          # 정답이 빈 행은 지운다
# df['열이름'] = df['열이름'].fillna(0)                      # '없음'이 0을 뜻할 때 (예: 강수량)
# df['열이름'] = df['열이름'].fillna(df['열이름'].median())    # 숫자: 중앙값으로
# df['열이름'] = df['열이름'].fillna('없음')                  # 글자: '없음'이라는 범주로 (9차시 자격증)
print('남은 결측치:', int(df.isna().sum().sum()))

## 3-3. 중복 행

In [ ]:
print('중복 행 수:', df.duplicated().sum())
# df = df.drop_duplicates()   # 🔧 중복을 지워야 하는 데이터라면

## 3-4. 정답(y)의 분포

In [ ]:
if TASK == '회귀':
    df[TARGET].plot(kind='hist', bins=30, color='#2A9D8F', title=f'{TARGET} 분포')
    plt.show()
    print(df[TARGET].describe().round(2))
else:
    counts = df[TARGET].value_counts()
    counts.plot(kind='bar', color='#E76F51', title=f'{TARGET} 비율')
    plt.show()
    print((counts / counts.sum()).round(3))
    print('⚠️ 가장 많은 쪽만 찍어도 정확도', round(counts.max() / counts.sum(), 3), '→ 이 숫자가 기준선이다 (9차시)')

## 3-5. 이상치 — 숫자 열의 상자 그림

In [ ]:
num_cols = df.select_dtypes('number').columns.drop(TARGET, errors='ignore')
if len(num_cols):
    (df[num_cols] - df[num_cols].mean()).div(df[num_cols].std()).plot(kind='box', rot=45, title='숫자 열 분포 (표준화해서 한 그림에)')
    plt.show()
# 🔧 이상치를 지우거나 고칠 때는 '왜 이상한지'를 먼저 확인하고, 처리 이유를 기록한다

## 3-6. 정답과의 관계 — 어떤 특성이 관계가 깊을까?

In [ ]:
num = df.select_dtypes('number')
if TASK == '회귀' and TARGET in num:
    corr = num.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)
    plt.title('상관계수 히트맵')
    plt.show()
    print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3))
else:
    for c in num.columns.drop(TARGET, errors='ignore')[:6]:
        print(df.groupby(TARGET)[c].mean().round(2).rename(f'{c} 평균'), '\n')

In [ ]:
# 🔧 글자(범주) 열과 정답의 관계 — 열 이름을 골라 본다
cat_cols = [c for c in df.select_dtypes(exclude='number').columns if c != TARGET]
for c in cat_cols[:4]:
    if TASK == '회귀':
        df.groupby(c)[TARGET].mean().sort_values().plot(kind='barh', title=f'{c}별 평균 {TARGET}')
        plt.show()
    else:
        print(pd.crosstab(df[c], df[TARGET], normalize='index').round(2), '\n')

## 3-7. 파생 변수 만들기 (선택)

In [ ]:
# 🔧 예: 날짜에서 월·요일 꺼내기, 두 열을 더하거나 나누기, 구간으로 나누기
# df['파생열'] = df['열1'] / df['열2']
print('현재 열:', list(df.columns))

✍️ **우리 팀의 기록** — 탐색에서 발견한 것 세 가지(정답과 관계 깊은 특성, 놀라운 점, 이상한 점)와 전처리에서 한 일·이유를 적는다.

> (여기에 적기)

---
# ④ 모델 생성 및 학습
## 4-1. X·y 준비와 학습용/평가용 나누기
분류라면 `stratify`로 두 묶음의 정답 비율을 똑같이 맞춘다 (6·9차시).

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if TASK == '분류' else None)
print('학습용:', len(X_train), '/ 평가용:', len(X_test))

## 4-2. 기준선 — 아무것도 배우지 않은 모델 (회귀: 평균으로 찍기 / 분류: 가장 많은 쪽으로 찍기)

In [ ]:
baseline = (DummyRegressor(strategy='mean') if TASK == '회귀' else DummyClassifier(strategy='most_frequent'))
baseline.fit(X_train, y_train)
base_score = score(baseline, X_test, y_test)
print('기준선 점수:', base_score)

## 4-3. 첫 모델 학습 — 회귀: 선형 회귀 / 분류: 로지스틱 회귀

In [ ]:
model = build_model('linear', FEATURES, df)
model.fit(X_train, y_train)
if TASK == '회귀':
    coef = pd.Series(model[-1].coef_, index=[n.split('__')[-1] for n in model[0].get_feature_names_out()]).round(3)
    print('특성별 기울기 (표준화 단위 · 클수록 영향이 큼):'); print(coef)
else:
    print('학습 완료 · 클래스:', list(model[-1].classes_))

✍️ **우리 팀의 기록** — 우리가 고른 모델과 그 이유(정답의 종류·데이터 크기·해석 가능성)를 적는다.

> (여기에 적기)

---
# ⑤ 모델 평가
처음 보는 평가용 데이터로 점수를 매기고, **기준선과 비교**한다.

In [ ]:
first_score = score(model, X_test, y_test)
print('첫 모델:', first_score)
print('기준선 :', base_score)

In [ ]:
pred = model.predict(X_test)
if TASK == '회귀':
    plt.scatter(y_test, pred, alpha=0.6)
    lo, hi = y_test.min(), y_test.max()
    plt.plot([lo, hi], [lo, hi], color='red')
    plt.title('실제 vs 예측'); plt.xlabel('실제'); plt.ylabel('예측')
    plt.show()
else:
    labels = sorted(y.unique())
    sns.heatmap(confusion_matrix(y_test, pred, labels=labels), annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'예측:{l}' for l in labels], yticklabels=[f'실제:{l}' for l in labels])
    plt.title('혼동 행렬 (행 = 실제, 열 = 예측)')
    plt.show()
    print(classification_report(y_test, pred))

✍️ **우리 팀의 기록** — 점수는 기준선보다 얼마나 좋은가? 어디서 많이 틀리는가(산점도에서 벗어난 구간, 혼동 행렬의 칸)? 분류라면 어떤 실수가 더 문제인가?

> (여기에 적기)

---
# ⑥ 성능 개선
**한 번에 하나씩** 바꾸고, **교차 검증**으로 공정하게 비교한다 (8차시).
## 6-1. 특성 바꾸기

In [ ]:
# 🔧 비교하고 싶은 특성 조합을 적는다 (데이터에 있는 열 이름으로)
FEATURE_SETS = {
    '첫 모델': FEATURES,
    '+ 특성 추가': FEATURES + ['미세먼지'],
    '+ 범주 추가': FEATURES + ['주말여부', '계절'],
}
rows = []
for name, feats in FEATURE_SETS.items():
    feats = [f for f in feats if f in df.columns]
    rows.append({'특성 조합': name, '특성': ', '.join(feats), '교차 검증 점수': cv_score(build_model('linear', feats, df), df[feats], y)})
feature_table = pd.DataFrame(rows)
feature_table

## 6-2. 모델 바꾸기

In [ ]:
# 🔧 6-1에서 점수가 가장 높은 특성 조합을 자동으로 고른다
best_name = feature_table.sort_values('교차 검증 점수', ascending=False).iloc[0]['특성 조합']
BEST_FEATURES = [c for c in FEATURE_SETS[best_name] if c in df.columns]   # 직접 리스트로 적어도 된다
candidates = ({'선형 회귀': 'linear', '다항 회귀(2차)': 'poly', '결정 트리(깊이 4)': 'tree'} if TASK == '회귀'
              else {'로지스틱 회귀': 'linear', '로지스틱(불균형 보정)': 'linear_balanced', '결정 트리(깊이 4)': 'tree'})
rows = [{'모델': name, '교차 검증 점수': cv_score(build_model(alg, BEST_FEATURES, df), df[BEST_FEATURES], y)}
        for name, alg in candidates.items()]
model_table = pd.DataFrame(rows)
print('사용 특성:', BEST_FEATURES)
model_table

## 6-3. 최종 모델 고르기 — 점수만이 아니라 **단순함·설명 가능성**도 함께 본다

In [ ]:
# 🔧 최종 모델: 기본은 6-2에서 점수가 가장 높은 모델. 더 단순한 모델을 고르려면 FINAL_ALG = 'linear' 처럼 직접 적는다
FINAL_ALG = candidates[model_table.sort_values('교차 검증 점수', ascending=False).iloc[0]['모델']]
final_model = build_model(FINAL_ALG, BEST_FEATURES, df)
final_model.fit(df.loc[X_train.index, BEST_FEATURES], y_train)   # 같은 학습용 행으로 학습
final_score = score(final_model, df.loc[X_test.index, BEST_FEATURES], y_test)
print('최종 모델:', FINAL_ALG, '/ 특성:', BEST_FEATURES)
print('평가용 점수 — 첫 모델:', first_score, '→ 최종 모델:', final_score, '/ 기준선:', base_score)

✍️ **우리 팀의 기록** — 무엇을 바꿨고 점수가 어떻게 변했나? 너무 좋은 점수가 나왔다면 정답 정보가 새어 들어온(데이터 누수) 특성은 없는지 확인했나? (9차시)

> (여기에 적기)

---
# ⑦ 모델 활용
## 7-1. 최종 모델을 전체 데이터로 다시 학습하고, 새 데이터를 예측한다

In [ ]:
final_model.fit(df[BEST_FEATURES], y)
# 🔧 예측해 보고 싶은 새 데이터 — 학습 데이터의 첫 행을 바탕으로 값을 바꿔 쓴다
new_data = df[BEST_FEATURES].iloc[[0]].copy()
# new_data['평균기온'] = 25.0      # 예: 값 바꾸기
print('입력:'); display(new_data)
print('예측:', final_model.predict(new_data))
if TASK == '분류':
    print('클래스별 확률:', dict(zip(final_model[-1].classes_, final_model.predict_proba(new_data)[0].round(3))))

## 7-2. 모델 저장 — 대시보드(웹앱)에서 불러 쓸 수 있다

In [ ]:
import joblib
joblib.dump(final_model, f'{TEAM}_model.joblib')
print('저장 완료:', f'{TEAM}_model.joblib')

✍️ **활용 제안과 한계**

| 항목 | 내용 |
|---|---|
| 누가, 어떻게 쓸까 |  |
| 이렇게 쓰면 안 된다 |  |
| 데이터의 한계 (양·기간·대표성) |  |
| 모델의 한계 (외삽·상관≠인과·숨은 변수) |  |
| 윤리 점검 (개인정보·편향·설명 가능성) |  |

---
# ⑧ 보고서용 요약 — 아래 표를 보고서 템플릿에 옮긴다

In [ ]:
summary = pd.DataFrame({
    '항목': ['팀', '데이터', '정답(y) · 문제 종류', '첫 특성', '최종 특성', '최종 모델',
             '기준선 점수', '첫 모델 점수', '최종 모델 점수'],
    '내용': [TEAM, f'{DATA_FILE} ({df.shape[0]}행 × {df.shape[1]}열)', f'{TARGET} · {TASK}', ', '.join(FEATURES),
             ', '.join(BEST_FEATURES), FINAL_ALG, str(base_score), str(first_score), str(final_score)]})
summary

### ✅ 제출 전 체크리스트
- [ ] 위에서부터 **처음부터 다시 실행**해도 오류가 없다 (런타임 → 모두 실행)
- [ ] 모든 `✍️ 우리 팀의 기록` 칸을 채웠다
- [ ] 그래프마다 제목·축 이름이 있다
- [ ] 데이터 출처를 기록했다
- [ ] 보고서 템플릿에 ⑧ 요약표와 핵심 그래프를 옮겼다